# Cuadro de mandos por etapas · TFG IHTC

Notebook para ejecutar, validar y visualizar las dos primeras fases:

1. **Fase 1**: admisiones y quirófanos.
2. **Fase 2**: habitaciones y feedback hacia fase 1.
3. **Validador oficial**: usa exclusivamente `IHTP_Validator.exe`.
4. **Dashboard interno**: métricas, tablas y gráficos.

Este notebook está preparado para la carpeta:

```text
C:\Users\angel\OneDrive\Escritorio\tfg
```

In [1]:
%pip install -U nbformat plotly ipykernel

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\angel\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [2]:
# ============================================================
# 0. IMPORTS Y CONFIGURACIÓN GENERAL
# ============================================================

from pathlib import Path
import json
import shutil
import subprocess
import re
import importlib.util
import os
from datetime import datetime

import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display, Markdown

pio.renderers.default = "vscode"# -----------------------------
# RUTA PRINCIPAL DEL TFG
# -----------------------------

BASE_DIR = Path(r"C:\Users\angel\OneDrive\Escritorio\tfg")

if not BASE_DIR.exists():
    BASE_DIR = Path.cwd()
    print("[AVISO] No se encontró la ruta configurada. Se usará la carpeta actual:")
    print(BASE_DIR)

INSTANCE_PATH = BASE_DIR / "test01.json"
SCRIPT_FASE1 = BASE_DIR / "modelo_scp_gurobi.py"
SCRIPT_FASE2 = BASE_DIR / "modelo_habitaciones_gurobi.py"

VALIDATOR_EXE = BASE_DIR / "IHTP_Validator.exe"
VALIDATOR_SRC = BASE_DIR / "IHTP_Validator.cc"

RESULTS_DIR = BASE_DIR / "resultados_dashboard_notebook"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# PARÁMETROS DE EJECUCIÓN
# -----------------------------

N_ITERACIONES = 2

TIME_LIMIT_FASE1 = 120
MIP_GAP_FASE1 = 0.02
TIME_LIMIT_FASE2 = 60

# Si quieres borrar resultados anteriores antes de ejecutar, cambia a True.
RESET_RESULTS = False

if RESET_RESULTS and RESULTS_DIR.exists():
    shutil.rmtree(RESULTS_DIR)
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("Instancia:", INSTANCE_PATH, "| existe:", INSTANCE_PATH.exists())
print("Script fase 1:", SCRIPT_FASE1, "| existe:", SCRIPT_FASE1.exists())
print("Script fase 2:", SCRIPT_FASE2, "| existe:", SCRIPT_FASE2.exists())
print("Validador exe:", VALIDATOR_EXE, "| existe:", VALIDATOR_EXE.exists())
print("Validador fuente:", VALIDATOR_SRC, "| existe:", VALIDATOR_SRC.exists())
print("Carpeta resultados:", RESULTS_DIR)

BASE_DIR: C:\Users\angel\OneDrive\Escritorio\tfg
Instancia: C:\Users\angel\OneDrive\Escritorio\tfg\test01.json | existe: True
Script fase 1: C:\Users\angel\OneDrive\Escritorio\tfg\modelo_scp_gurobi.py | existe: True
Script fase 2: C:\Users\angel\OneDrive\Escritorio\tfg\modelo_habitaciones_gurobi.py | existe: True
Validador exe: C:\Users\angel\OneDrive\Escritorio\tfg\IHTP_Validator.exe | existe: True
Validador fuente: C:\Users\angel\OneDrive\Escritorio\tfg\IHTP_Validator.cc | existe: True
Carpeta resultados: C:\Users\angel\OneDrive\Escritorio\tfg\resultados_dashboard_notebook


## 1. Utilidades seguras

In [3]:
# ============================================================
# 1. UTILIDADES
# ============================================================

def load_json(path):
    path = Path(path)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)


def import_module_from_path(module_name, path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"No se encuentra el archivo requerido: {path}")

    spec = importlib.util.spec_from_file_location(module_name, str(path))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def path_readable(path, base=None):
    path = Path(path)
    if base is None:
        base = BASE_DIR
    try:
        return str(path.resolve().relative_to(base.resolve()))
    except Exception:
        return str(path.resolve())


def feedback_is_empty(path):
    """True si el feedback no existe o si no contiene penalizaciones/caps."""
    path = Path(path)
    if not path.exists():
        return True

    try:
        fb = load_json(path)
    except Exception:
        return True

    for key in ["day_penalties", "day_admission_caps"]:
        value = fb.get(key, {})
        if isinstance(value, dict) and len(value) > 0:
            return False

    for key in ["gender_day_penalties", "gender_day_admission_caps"]:
        value = fb.get(key, {})
        if isinstance(value, dict):
            for inner in value.values():
                if isinstance(inner, dict) and len(inner) > 0:
                    return False

    return True


def flatten_dict(d, parent_key="", sep="."):
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else str(k)
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)

## 2. Preparar el validador oficial

In [4]:
# ============================================================
# 2. PREPARAR VALIDADOR
# ============================================================
# Este notebook NO ejecuta el archivo 'validador' sin extensión.
# Solo usa IHTP_Validator.exe.

def posibles_gpp():
    candidatos = []

    gpp_path = shutil.which("g++")
    if gpp_path:
        candidatos.append(Path(gpp_path))

    candidatos.extend([
        Path(r"C:\msys64\ucrt64\bin\g++.exe"),
        Path(r"C:\msys64\mingw64\bin\g++.exe"),
        Path(r"C:\Program Files\MSYS2\ucrt64\bin\g++.exe"),
        Path(r"C:\Program Files\MSYS2\mingw64\bin\g++.exe"),
        Path(r"C:\Program Files (x86)\MSYS2\ucrt64\bin\g++.exe"),
        Path(r"C:\Program Files (x86)\MSYS2\mingw64\bin\g++.exe"),
    ])

    out = []
    seen = set()
    for c in candidatos:
        s = str(c)
        if s not in seen:
            out.append(c)
            seen.add(s)
    return out


def compilar_validador_si_es_necesario():
    if VALIDATOR_EXE.exists():
        return {"ok": True, "message": "El ejecutable ya existe.", "exe": str(VALIDATOR_EXE), "compiler": None, "stdout": "", "stderr": ""}

    if not VALIDATOR_SRC.exists():
        return {"ok": False, "message": "No existe IHTP_Validator.exe ni IHTP_Validator.cc.", "exe": str(VALIDATOR_EXE), "compiler": None, "stdout": "", "stderr": ""}

    last = {"ok": False, "message": "No se encontró g++ en rutas conocidas.", "exe": str(VALIDATOR_EXE), "compiler": None, "stdout": "", "stderr": ""}

    for compiler in posibles_gpp():
        if not compiler.exists():
            continue

        cmd = [str(compiler), "-std=c++17", "-O2", str(VALIDATOR_SRC), "-o", str(VALIDATOR_EXE)]

        try:
            res = subprocess.run(cmd, cwd=str(BASE_DIR), capture_output=True, text=True, timeout=120)
            if res.returncode == 0 and VALIDATOR_EXE.exists():
                return {"ok": True, "message": "Validador compilado correctamente.", "exe": str(VALIDATOR_EXE), "compiler": str(compiler), "stdout": res.stdout, "stderr": res.stderr}

            last = {"ok": False, "message": "Intento de compilación fallido.", "exe": str(VALIDATOR_EXE), "compiler": str(compiler), "stdout": res.stdout, "stderr": res.stderr}
        except Exception as e:
            last = {"ok": False, "message": "Excepción compilando.", "exe": str(VALIDATOR_EXE), "compiler": str(compiler), "stdout": "", "stderr": repr(e)}

    return last


estado_validador = compilar_validador_si_es_necesario()
display(pd.DataFrame([estado_validador]))

print("VALIDATOR_EXE:", VALIDATOR_EXE)
print("Existe:", VALIDATOR_EXE.exists())

,ok,message,exe,compiler,stdout,stderr
0,True,El ejecutable ya existe.,C:\Users\angel\OneDrive\Escritorio\tfg\IHTP_Va...,None,,


VALIDATOR_EXE: C:\Users\angel\OneDrive\Escritorio\tfg\IHTP_Validator.exe
Existe: True


## 3. Importar scripts y cargar instancia

In [5]:
# ============================================================
# 3. IMPORTAR SCRIPTS Y CARGAR INSTANCIA
# ============================================================

errores_iniciales = []

for required in [INSTANCE_PATH, SCRIPT_FASE1, SCRIPT_FASE2]:
    if not required.exists():
        errores_iniciales.append(f"No existe: {required}")

if errores_iniciales:
    for e in errores_iniciales:
        print("[ERROR]", e)
    raise FileNotFoundError("Faltan archivos requeridos. Revisa BASE_DIR.")

fase1 = import_module_from_path("modelo_scp_gurobi", SCRIPT_FASE1)
fase2 = import_module_from_path("modelo_habitaciones_gurobi", SCRIPT_FASE2)

instancia = load_json(INSTANCE_PATH)

display(Markdown(f"""
### Instancia cargada: `{INSTANCE_PATH.name}`

- Días del horizonte: **{instancia["days"]}**
- Turnos diarios: **{", ".join(instancia.get("shift_types", []))}**
- Pacientes candidatos: **{len(instancia["patients"])}**
- Pacientes obligatorios: **{sum(1 for p in instancia["patients"] if p.get("mandatory"))}**
- Pacientes opcionales: **{sum(1 for p in instancia["patients"] if not p.get("mandatory"))}**
- Ocupantes iniciales: **{len(instancia.get("occupants", []))}**
- Habitaciones: **{len(instancia["rooms"])}**
- Camas totales: **{sum(r["capacity"] for r in instancia["rooms"])}**
- Quirófanos: **{len(instancia["operating_theaters"])}**
- Cirujanos: **{len(instancia["surgeons"])}**
- Enfermeras: **{len(instancia["nurses"])}**
"""))

>>> Ejecutando modelo_habitaciones_gurobi.py



### Instancia cargada: `test01.json`

- Días del horizonte: **21**
- Turnos diarios: **early, late, night**
- Pacientes candidatos: **42**
- Pacientes obligatorios: **11**
- Pacientes opcionales: **31**
- Ocupantes iniciales: **7**
- Habitaciones: **5**
- Camas totales: **13**
- Quirófanos: **2**
- Cirujanos: **1**
- Enfermeras: **13**


## 4. Ejecutar fase 1 y fase 2

In [6]:
# ============================================================
# 4. EJECUCIÓN DE FASES
# ============================================================

def ejecutar_iteracion(i, feedback_entrada=None):
    out_dir = RESULTS_DIR / f"iter_{i:02d}"
    out_dir.mkdir(parents=True, exist_ok=True)

    sol_fase1 = out_dir / "solucion_fase1.json"
    sol_fase2 = out_dir / "solucion_fase2.json"
    feedback_salida = out_dir / "feedback_fase2.json"

    for p in [sol_fase1, sol_fase2, feedback_salida]:
        if p.exists():
            p.unlink()

    print("\n" + "=" * 80)
    print(f"ITERACIÓN {i}")
    print("=" * 80)

    feedback_path = None
    if feedback_entrada is not None and Path(feedback_entrada).exists() and not feedback_is_empty(feedback_entrada):
        feedback_path = str(feedback_entrada)
        print(f"[INFO] Fase 1 con feedback: {feedback_path}")
    else:
        print("[INFO] Fase 1 sin feedback.")

    instancia_local = fase1.cargar_instancia(str(INSTANCE_PATH))

    try:
        fase1.resolver_fase1_scp_interactiva(
            instancia_local,
            ruta_salida=str(sol_fase1),
            ruta_feedback=feedback_path,
            time_limit=TIME_LIMIT_FASE1,
            mip_gap=MIP_GAP_FASE1,
            usar_capacidad_genero=True,
        )
    except Exception as e:
        print("[ERROR] Falló fase 1:", repr(e))

    fase1_ok = sol_fase1.exists()

    if not fase1_ok:
        return {"iteracion": i, "fase1_ok": False, "fase2_ok": False, "sol_fase1": sol_fase1, "sol_fase2": sol_fase2, "feedback": feedback_salida, "feedback_usado": feedback_path}

    try:
        fase2.resolver_habitaciones_debug(
            ruta_instancia=str(INSTANCE_PATH),
            ruta_sol_previa=str(sol_fase1),
            ruta_final=str(sol_fase2),
            ruta_feedback=str(feedback_salida),
            time_limit=TIME_LIMIT_FASE2,
        )
    except Exception as e:
        print("[ERROR] Falló fase 2:", repr(e))

    fase2_ok = sol_fase2.exists()

    return {"iteracion": i, "fase1_ok": fase1_ok, "fase2_ok": fase2_ok, "sol_fase1": sol_fase1, "sol_fase2": sol_fase2, "feedback": feedback_salida, "feedback_usado": feedback_path}


resultados_iteraciones = []
feedback_actual = None

for i in range(N_ITERACIONES):
    res = ejecutar_iteracion(i, feedback_actual)
    resultados_iteraciones.append(res)

    if res["feedback"].exists() and not feedback_is_empty(res["feedback"]):
        feedback_actual = res["feedback"]
    else:
        feedback_actual = None


df_iteraciones = pd.DataFrame([
    {
        "iteración": r["iteracion"],
        "fase1_ok": r["fase1_ok"],
        "fase2_ok": r["fase2_ok"],
        "feedback_usado": r["feedback_usado"],
        "sol_fase1": path_readable(r["sol_fase1"]) if r["sol_fase1"].exists() else None,
        "sol_fase2": path_readable(r["sol_fase2"]) if r["sol_fase2"].exists() else None,
        "feedback": path_readable(r["feedback"]) if r["feedback"].exists() else None,
        "feedback_vacío": feedback_is_empty(r["feedback"]),
    }
    for r in resultados_iteraciones
])

df_iteraciones


ITERACIÓN 0
[INFO] Fase 1 sin feedback.

--- FASE 1A: Admisión de pacientes + anticipación de camas y quirófanos ---
Set parameter Username
Set parameter LicenseID to value 2810787
Academic license - for non-commercial use only - expires 2027-04-20
Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 120
Set parameter MIPGap to value 0.02
Optimizando fase 1A...
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  120
MIPGap  0.02

Optimize a model with 944 rows, 886 columns and 8945 nonzeros (Min)
Model fingerprint: 0x38b9840d
Model has 31 linear objective coefficients
Variable types: 0 continuous, 886 integer (886 binary)
Coefficient statistics:
  Matrix range     [1e+00, 7e+02]
  Objective range  [1e-03, 2e+02]
  Bounds range     

,iteración,fase1_ok,fase2_ok,feedback_usado,sol_fase1,sol_fase2,feedback,feedback_vacío
0,0,True,True,None,resultados_dashboard_notebook\iter_00\solucion...,resultados_dashboard_notebook\iter_00\solucion...,resultados_dashboard_notebook\iter_00\feedback...,True
1,1,True,True,None,resultados_dashboard_notebook\iter_01\solucion...,resultados_dashboard_notebook\iter_01\solucion...,resultados_dashboard_notebook\iter_01\feedback...,True


## 5. Validar soluciones con el validador oficial

In [7]:
# ============================================================
# 5. VALIDACIÓN
# ============================================================

def parsear_salida_validador(texto):
    violations = {}
    costs = {}
    total_violations = None
    total_cost = None
    seccion = None

    for raw_line in texto.splitlines():
        line = raw_line.strip()

        if not line:
            continue

        if line.startswith("VIOLATIONS"):
            seccion = "violations"
            continue

        if line.startswith("COSTS"):
            seccion = "costs"
            continue

        if line.startswith("Total violations"):
            m = re.search(r"Total violations\s*=\s*([0-9]+)", line)
            if m:
                total_violations = int(m.group(1))
            continue

        if line.startswith("Total cost"):
            m = re.search(r"Total cost\s*=\s*([0-9]+)", line)
            if m:
                total_cost = int(m.group(1))
            continue

        if seccion == "violations":
            m = re.match(r"([A-Za-z]+)\.*\s*([0-9]+)", line)
            if m:
                violations[m.group(1)] = int(m.group(2))

        if seccion == "costs":
            m = re.match(r"([A-Za-z]+)\.*\s*([0-9]+)\s*\(\s*([0-9]+)\s*X\s*([0-9]+)\s*\)", line)
            if m:
                costs[m.group(1)] = {"weighted_cost": int(m.group(2)), "weight": int(m.group(3)), "raw_cost": int(m.group(4))}

    return {"violations": violations, "costs": costs, "total_violations": total_violations, "total_cost": total_cost}


def ejecutar_validador(ruta_instancia, ruta_solucion, verbose=False):
    ruta_instancia = Path(ruta_instancia)
    ruta_solucion = Path(ruta_solucion)

    salida = {"ok": False, "stdout": "", "stderr": "", "violations": {}, "costs": {}, "total_violations": None, "total_cost": None, "command": None}

    if not VALIDATOR_EXE.exists():
        salida["stderr"] = f"No existe IHTP_Validator.exe en {VALIDATOR_EXE}"
        return salida

    if not ruta_instancia.exists():
        salida["stderr"] = f"No existe la instancia: {ruta_instancia}"
        return salida

    if not ruta_solucion.exists():
        salida["stderr"] = f"No existe la solución: {ruta_solucion}"
        return salida

    cmd = [str(VALIDATOR_EXE), str(ruta_instancia), str(ruta_solucion)]

    if verbose:
        cmd.append("verbose")

    salida["command"] = " ".join(cmd)

    try:
        res = subprocess.run(cmd, cwd=str(BASE_DIR), capture_output=True, text=True, timeout=120, shell=False)
        stdout = res.stdout or ""
        stderr = res.stderr or ""
        parsed = parsear_salida_validador(stdout + "\n" + stderr)

        salida.update({"ok": res.returncode == 0, "stdout": stdout, "stderr": stderr, "violations": parsed["violations"], "costs": parsed["costs"], "total_violations": parsed["total_violations"], "total_cost": parsed["total_cost"]})
        return salida

    except Exception as e:
        salida["stderr"] = repr(e)
        return salida


validaciones = []

for r in resultados_iteraciones:
    for fase, ruta_sol in [("fase1", r["sol_fase1"]), ("fase2", r["sol_fase2"])]:
        val = ejecutar_validador(INSTANCE_PATH, ruta_sol)

        fila = {"iteración": r["iteracion"], "fase": fase, "solución": path_readable(ruta_sol) if Path(ruta_sol).exists() else None, "ok_validador": val["ok"], "total_violations": val["total_violations"], "total_cost": val["total_cost"], "stderr": val["stderr"], "command": val["command"]}

        for k, v in val["violations"].items():
            fila[f"viol_{k}"] = v

        for k, v in val["costs"].items():
            fila[f"cost_{k}"] = v["weighted_cost"]
            fila[f"raw_{k}"] = v["raw_cost"]

        validaciones.append(fila)

df_validaciones = pd.DataFrame(validaciones)
df_validaciones

,iteración,fase,solución,ok_validador,total_violations,total_cost,stderr,command,viol_RoomGenderMix,viol_PatientRoomCompatibility,...,cost_ExcessiveNurseWorkload,raw_ExcessiveNurseWorkload,cost_OpenOperatingTheater,raw_OpenOperatingTheater,cost_SurgeonTransfer,raw_SurgeonTransfer,cost_PatientDelay,raw_PatientDelay,cost_ElectiveUnscheduledPatients,raw_ElectiveUnscheduledPatients
0,0,fase1,resultados_dashboard_notebook\iter_00\solucion...,False,NaN,NaN,terminate called after throwing an instance of...,C:\Users\angel\OneDrive\Escritorio\tfg\IHTP_Va...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,fase2,resultados_dashboard_notebook\iter_00\solucion...,True,252.0,2095.0,,C:\Users\angel\OneDrive\Escritorio\tfg\IHTP_Va...,0.0,0.0,...,0.0,0.0,330.0,11.0,0.0,0.0,460.0,92.0,1200.0,8.0
2,1,fase1,resultados_dashboard_notebook\iter_01\solucion...,False,NaN,NaN,terminate called after throwing an instance of...,C:\Users\angel\OneDrive\Escritorio\tfg\IHTP_Va...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,fase2,resultados_dashboard_notebook\iter_01\solucion...,True,252.0,2095.0,,C:\Users\angel\OneDrive\Escritorio\tfg\IHTP_Va...,0.0,0.0,...,0.0,0.0,330.0,11.0,0.0,0.0,460.0,92.0,1200.0,8.0


In [8]:
pd.set_option("display.max_colwidth", None)

sol = RESULTS_DIR / "iter_01" / "solucion_fase2.json"

cmd = [
    str(VALIDATOR_EXE),
    str(INSTANCE_PATH),
    str(sol)
]

print("EXE existe:", VALIDATOR_EXE.exists())
print("Solución existe:", sol.exists())
print("Comando:")
print(" ".join(cmd))

res = subprocess.run(
    cmd,
    cwd=str(BASE_DIR),
    capture_output=True,
    text=True,
    timeout=120,
    shell=False
)

print("\nRETURN CODE:")
print(res.returncode)

print("\nSTDOUT COMPLETO:")
print(res.stdout)

print("\nSTDERR COMPLETO:")
print(res.stderr)

EXE existe: True
Solución existe: True
Comando:
C:\Users\angel\OneDrive\Escritorio\tfg\IHTP_Validator.exe C:\Users\angel\OneDrive\Escritorio\tfg\test01.json C:\Users\angel\OneDrive\Escritorio\tfg\resultados_dashboard_notebook\iter_01\solucion_fase2.json

RETURN CODE:
0

STDOUT COMPLETO:
VIOLATIONS: 
RoomGenderMix.....................0
PatientRoomCompatibility..........0
SurgeonOvertime...................0
OperatingTheaterOvertime..........0
MandatoryUnscheduledPatients......0
AdmissionDay......................0
RoomCapacity......................0
NursePresence.....................0
UncoveredRoom...................252
Total violations = 252

COSTS (weight X cost): 
RoomAgeMix...........................105 (  5 X  21)
RoomSkillLevel.........................0 (  1 X   0)
ContinuityOfCare.......................0 (  5 X   0)
ExcessiveNurseWorkload.................0 (  1 X   0)
OpenOperatingTheater.................330 ( 30 X  11)
SurgeonTransfer........................0 (  1 X   0)
PatientDe

In [9]:
# Diagnóstico del validador sobre la última solución de fase 2 disponible.

ultima_sol_f2 = None
for r in reversed(resultados_iteraciones):
    if r["sol_fase2"].exists():
        ultima_sol_f2 = r["sol_fase2"]
        break

if ultima_sol_f2 is None:
    print("No hay solución de fase 2 generada para diagnosticar.")
else:
    val_prueba = ejecutar_validador(INSTANCE_PATH, ultima_sol_f2)
    print("Comando:", val_prueba["command"])
    print("OK:", val_prueba["ok"])
    print("Total violations:", val_prueba["total_violations"])
    print("Total cost:", val_prueba["total_cost"])
    print("\nSTDERR:")
    print(val_prueba["stderr"])
    print("\nSTDOUT:")
    print(val_prueba["stdout"])

Comando: C:\Users\angel\OneDrive\Escritorio\tfg\IHTP_Validator.exe C:\Users\angel\OneDrive\Escritorio\tfg\test01.json C:\Users\angel\OneDrive\Escritorio\tfg\resultados_dashboard_notebook\iter_01\solucion_fase2.json
OK: True
Total violations: 252
Total cost: 2095

STDERR:


STDOUT:
VIOLATIONS: 
RoomGenderMix.....................0
PatientRoomCompatibility..........0
SurgeonOvertime...................0
OperatingTheaterOvertime..........0
MandatoryUnscheduledPatients......0
AdmissionDay......................0
RoomCapacity......................0
NursePresence.....................0
UncoveredRoom...................252
Total violations = 252

COSTS (weight X cost): 
RoomAgeMix...........................105 (  5 X  21)
RoomSkillLevel.........................0 (  1 X   0)
ContinuityOfCare.......................0 (  5 X   0)
ExcessiveNurseWorkload.................0 (  1 X   0)
OpenOperatingTheater.................330 ( 30 X  11)
SurgeonTransfer........................0 (  1 X   0)
PatientDelay...

## 6. Métricas internas del cuadro de mandos

In [10]:
# ============================================================
# 6. MÉTRICAS INTERNAS
# ============================================================

def get_patient_dict(instancia):
    return {p["id"]: p for p in instancia["patients"]}


def age_map(instancia):
    return {age: i for i, age in enumerate(instancia.get("age_groups", []))}


def construir_ocupacion(instancia, solucion, incluir_ocupantes=True):
    pacientes_inst = get_patient_dict(instancia)
    age_value = age_map(instancia)
    rows = []

    for ps in solucion.get("patients", []):
        pid = ps["id"]
        if pid not in pacientes_inst:
            continue

        p = pacientes_inst[pid]
        adm = ps["admission_day"]
        room = ps.get("room")
        los = p["length_of_stay"]

        for d in range(adm, min(adm + los, instancia["days"])):
            rows.append({"tipo": "patient", "id": pid, "day": d, "room": room, "gender": p["gender"], "age_group": p["age_group"], "age_value": age_value.get(p["age_group"])})

    if incluir_ocupantes:
        for oc in instancia.get("occupants", []):
            for d in range(min(oc["length_of_stay"], instancia["days"])):
                rows.append({"tipo": "occupant", "id": oc["id"], "day": d, "room": oc["room_id"], "gender": oc["gender"], "age_group": oc["age_group"], "age_value": age_value.get(oc["age_group"])})

    return pd.DataFrame(rows)


def metricas_solucion(instancia, solucion):
    pacientes_inst = get_patient_dict(instancia)
    patients_sol = solucion.get("patients", [])

    scheduled = set(p["id"] for p in patients_sol)
    mandatory = set(p["id"] for p in instancia["patients"] if p.get("mandatory"))
    optional = set(p["id"] for p in instancia["patients"] if not p.get("mandatory"))

    df_occ = construir_ocupacion(instancia, solucion, incluir_ocupantes=True)
    cap = {r["id"]: r["capacity"] for r in instancia["rooms"]}

    room_day_rows = []
    if not df_occ.empty:
        for (room, day), g in df_occ.groupby(["room", "day"]):
            genders = sorted(g["gender"].dropna().unique().tolist())
            ages = g["age_value"].dropna().tolist()
            occupancy = len(g)
            capacity = cap.get(room, 0)

            room_day_rows.append({"room": room, "day": day, "occupancy": occupancy, "capacity": capacity, "excess_capacity": max(0, occupancy - capacity), "gender_mix": int(len(genders) > 1), "genders": ",".join(genders), "age_mix": (max(ages) - min(ages)) if len(ages) > 0 else 0})

    df_room_day = pd.DataFrame(room_day_rows)

    delay = 0
    open_ots = set()

    for ps in patients_sol:
        pid = ps["id"]
        if pid not in pacientes_inst:
            continue

        p = pacientes_inst[pid]
        d = ps["admission_day"]

        delay += d - p["surgery_release_day"]

        if ps.get("operating_theater") is not None:
            open_ots.add((ps.get("operating_theater"), d))

    metrics = {"scheduled_patients": len(scheduled), "unscheduled_optional": len(optional - scheduled), "unscheduled_mandatory": len(mandatory - scheduled), "total_delay_raw": delay, "open_ots_raw": len(open_ots), "room_capacity_excess_raw": int(df_room_day["excess_capacity"].sum()) if not df_room_day.empty else 0, "room_gender_mix_raw": int(df_room_day["gender_mix"].sum()) if not df_room_day.empty else 0, "room_age_mix_raw": int(df_room_day["age_mix"].sum()) if not df_room_day.empty else 0}

    return metrics, df_occ, df_room_day


def solution_label(iteracion, fase):
    return f"Iteración {iteracion} · {fase}"


soluciones = {}

for r in resultados_iteraciones:
    if r["sol_fase1"].exists():
        soluciones[(r["iteracion"], "fase1")] = load_json(r["sol_fase1"])
    if r["sol_fase2"].exists():
        soluciones[(r["iteracion"], "fase2")] = load_json(r["sol_fase2"])

resumen_metricas = []
room_day_tables = {}
occ_tables = {}

for (it, fase), sol in soluciones.items():
    metrics, df_occ, df_room_day = metricas_solucion(instancia, sol)
    resumen_metricas.append({"iteración": it, "fase": fase, **metrics})
    occ_tables[(it, fase)] = df_occ
    room_day_tables[(it, fase)] = df_room_day


df_metricas = pd.DataFrame(resumen_metricas)
df_metricas

,iteración,fase,scheduled_patients,unscheduled_optional,unscheduled_mandatory,total_delay_raw,open_ots_raw,room_capacity_excess_raw,room_gender_mix_raw,room_age_mix_raw
0,0,fase1,34,8,0,92,0,0,0,5
1,0,fase2,34,8,0,92,11,0,0,21
2,1,fase1,34,8,0,92,0,0,0,5
3,1,fase2,34,8,0,92,11,0,0,21


## 7. Visualizaciones

In [11]:
# Coste y violaciones del validador.

if not df_validaciones.empty:
    df_plot = df_validaciones.copy()
    df_plot["solución_label"] = "Iter " + df_plot["iteración"].astype(str) + " · " + df_plot["fase"].astype(str)

    if df_plot["total_cost"].notna().any():
        fig = px.bar(df_plot, x="solución_label", y="total_cost", title="Coste total según validador", labels={"solución_label": "Solución", "total_cost": "Coste total"})
        fig.show()

    if df_plot["total_violations"].notna().any():
        fig = px.bar(df_plot, x="solución_label", y="total_violations", title="Violaciones totales según validador", labels={"solución_label": "Solución", "total_violations": "Violaciones"})
        fig.show()

display(df_validaciones)

,iteración,fase,solución,ok_validador,total_violations,total_cost,stderr,command,viol_RoomGenderMix,viol_PatientRoomCompatibility,...,cost_ExcessiveNurseWorkload,raw_ExcessiveNurseWorkload,cost_OpenOperatingTheater,raw_OpenOperatingTheater,cost_SurgeonTransfer,raw_SurgeonTransfer,cost_PatientDelay,raw_PatientDelay,cost_ElectiveUnscheduledPatients,raw_ElectiveUnscheduledPatients
0,0,fase1,resultados_dashboard_notebook\iter_00\solucion_fase1.json,False,NaN,NaN,"terminate called after throwing an instance of 'nlohmann::json_abi_v3_11_3::detail::type_error'\n what(): [json.exception.type_error.302] type must be string, but is null\n",C:\Users\angel\OneDrive\Escritorio\tfg\IHTP_Validator.exe C:\Users\angel\OneDrive\Escritorio\tfg\test01.json C:\Users\angel\OneDrive\Escritorio\tfg\resultados_dashboard_notebook\iter_00\solucion_fase1.json,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,fase2,resultados_dashboard_notebook\iter_00\solucion_fase2.json,True,252.0,2095.0,,C:\Users\angel\OneDrive\Escritorio\tfg\IHTP_Validator.exe C:\Users\angel\OneDrive\Escritorio\tfg\test01.json C:\Users\angel\OneDrive\Escritorio\tfg\resultados_dashboard_notebook\iter_00\solucion_fase2.json,0.0,0.0,...,0.0,0.0,330.0,11.0,0.0,0.0,460.0,92.0,1200.0,8.0
2,1,fase1,resultados_dashboard_notebook\iter_01\solucion_fase1.json,False,NaN,NaN,"terminate called after throwing an instance of 'nlohmann::json_abi_v3_11_3::detail::type_error'\n what(): [json.exception.type_error.302] type must be string, but is null\n",C:\Users\angel\OneDrive\Escritorio\tfg\IHTP_Validator.exe C:\Users\angel\OneDrive\Escritorio\tfg\test01.json C:\Users\angel\OneDrive\Escritorio\tfg\resultados_dashboard_notebook\iter_01\solucion_fase1.json,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,fase2,resultados_dashboard_notebook\iter_01\solucion_fase2.json,True,252.0,2095.0,,C:\Users\angel\OneDrive\Escritorio\tfg\IHTP_Validator.exe C:\Users\angel\OneDrive\Escritorio\tfg\test01.json C:\Users\angel\OneDrive\Escritorio\tfg\resultados_dashboard_notebook\iter_01\solucion_fase2.json,0.0,0.0,...,0.0,0.0,330.0,11.0,0.0,0.0,460.0,92.0,1200.0,8.0


In [12]:
# Métricas internas fase 1 / fase 2.

if not df_metricas.empty:
    display(df_metricas.sort_values(["iteración", "fase"]))

    cols = ["scheduled_patients", "unscheduled_optional", "unscheduled_mandatory", "total_delay_raw", "open_ots_raw", "room_capacity_excess_raw", "room_gender_mix_raw", "room_age_mix_raw"]
    cols = [c for c in cols if c in df_metricas.columns]

    df_melt = df_metricas.melt(id_vars=["iteración", "fase"], value_vars=cols, var_name="métrica", value_name="valor")
    df_melt["solución"] = "Iter " + df_melt["iteración"].astype(str) + " · " + df_melt["fase"]

    fig = px.bar(df_melt, x="métrica", y="valor", color="solución", barmode="group", title="Comparación de métricas internas", labels={"métrica": "Métrica", "valor": "Valor"})
    fig.update_layout(xaxis_tickangle=-35)
    fig.show()
else:
    print("No hay métricas internas.")

,iteración,fase,scheduled_patients,unscheduled_optional,unscheduled_mandatory,total_delay_raw,open_ots_raw,room_capacity_excess_raw,room_gender_mix_raw,room_age_mix_raw
0,0,fase1,34,8,0,92,0,0,0,5
1,0,fase2,34,8,0,92,11,0,0,21
2,1,fase1,34,8,0,92,0,0,0,5
3,1,fase2,34,8,0,92,11,0,0,21


In [13]:
# Admisiones por día.

admission_rows = []

for (it, fase), sol in soluciones.items():
    for ps in sol.get("patients", []):
        admission_rows.append({"iteración": it, "fase": fase, "solución": solution_label(it, fase), "day": ps["admission_day"], "patient": ps["id"]})


df_adm = pd.DataFrame(admission_rows)

if not df_adm.empty:
    df_adm_count = df_adm.groupby(["solución", "day"]).size().reset_index(name="admisiones")
    fig = px.bar(df_adm_count, x="day", y="admisiones", color="solución", barmode="group", title="Admisiones por día", labels={"day": "Día", "admisiones": "Pacientes admitidos"})
    fig.show()
    display(df_adm_count)
else:
    print("No hay admisiones.")

,solución,day,admisiones
0,Iteración 0 · fase1,1,5
1,Iteración 0 · fase1,3,4
2,Iteración 0 · fase1,4,3
3,Iteración 0 · fase1,7,5
4,Iteración 0 · fase1,8,2
5,Iteración 0 · fase1,11,2
6,Iteración 0 · fase1,13,4
7,Iteración 0 · fase1,14,2
8,Iteración 0 · fase1,16,3
9,Iteración 0 · fase1,17,2


In [14]:
# Uso de quirófanos por día.

pacientes_inst = get_patient_dict(instancia)
ot_rows = []

for (it, fase), sol in soluciones.items():
    for ps in sol.get("patients", []):
        pid = ps["id"]
        if pid not in pacientes_inst:
            continue

        ot_rows.append({"iteración": it, "fase": fase, "solución": solution_label(it, fase), "day": ps["admission_day"], "operating_theater": ps.get("operating_theater"), "minutes": pacientes_inst[pid]["surgery_duration"]})


df_ot = pd.DataFrame(ot_rows)

if not df_ot.empty:
    df_ot_day = df_ot.groupby(["solución", "day"]).agg(minutos=("minutes", "sum"), quirofanos_abiertos=("operating_theater", lambda x: len(set(v for v in x if v is not None)))).reset_index()

    fig = px.bar(df_ot_day, x="day", y="minutos", color="solución", barmode="group", title="Minutos quirúrgicos por día", labels={"day": "Día", "minutos": "Minutos"})
    fig.show()

    fig = px.bar(df_ot_day, x="day", y="quirofanos_abiertos", color="solución", barmode="group", title="Quirófanos abiertos por día", labels={"day": "Día", "quirofanos_abiertos": "Quirófanos abiertos"})
    fig.show()

    display(df_ot_day)
else:
    print("No hay datos de quirófanos.")

,solución,day,minutos,quirofanos_abiertos
0,Iteración 0 · fase1,1,480,0
1,Iteración 0 · fase1,3,480,0
2,Iteración 0 · fase1,4,360,0
3,Iteración 0 · fase1,7,600,0
4,Iteración 0 · fase1,8,360,0
5,Iteración 0 · fase1,11,480,0
6,Iteración 0 · fase1,13,570,0
7,Iteración 0 · fase1,14,480,0
8,Iteración 0 · fase1,16,480,0
9,Iteración 0 · fase1,17,480,0


In [15]:
# Heatmap habitación x día para la última solución de fase 2 disponible.

key = None

for candidate in sorted(soluciones.keys(), reverse=True):
    if candidate[1] == "fase2":
        key = candidate
        break

if key is None and soluciones:
    key = sorted(soluciones.keys(), reverse=True)[0]

if key is not None and key in room_day_tables and not room_day_tables[key].empty:
    df_rd = room_day_tables[key]
    pivot_occ = df_rd.pivot(index="room", columns="day", values="occupancy").fillna(0)

    fig = px.imshow(pivot_occ, aspect="auto", title=f"Ocupación habitación × día · {solution_label(*key)}", labels={"x": "Día", "y": "Habitación", "color": "Ocupación"})
    fig.show()

    display(df_rd.sort_values(["day", "room"]))
else:
    print("No hay datos de ocupación por habitación.")

,room,day,occupancy,capacity,excess_capacity,gender_mix,genders,age_mix
0,r0,0,2,3,0,0,A,1
14,r1,0,2,2,0,0,B,0
32,r2,0,1,3,0,0,B,0
53,r3,0,1,3,0,0,A,0
72,r4,0,1,2,0,0,A,0
...,...,...,...,...,...,...,...,...
82,r4,19,1,2,0,0,B,0
13,r0,20,2,3,0,0,B,1
52,r2,20,2,3,0,0,A,0
71,r3,20,2,3,0,0,A,0


## 8. Feedback generado

In [16]:
feedback_rows = []

for r in resultados_iteraciones:
    fb_path = r["feedback"]

    if not fb_path.exists():
        feedback_rows.append({"iteración": r["iteracion"], "clave": "(no generado)", "valor": ""})
        continue

    fb = load_json(fb_path)
    flat = flatten_dict(fb)

    if not flat:
        feedback_rows.append({"iteración": r["iteracion"], "clave": "(feedback vacío)", "valor": ""})
    else:
        for k, v in flat.items():
            feedback_rows.append({"iteración": r["iteracion"], "clave": k, "valor": v})


df_feedback = pd.DataFrame(feedback_rows)
df_feedback

,iteración,clave,valor
0,0,(feedback vacío),
1,1,(feedback vacío),


## 9. Exportar tablas resumen

In [17]:
TABLAS_DIR = RESULTS_DIR / "tablas_resumen"
TABLAS_DIR.mkdir(parents=True, exist_ok=True)

df_iteraciones.to_csv(TABLAS_DIR / "iteraciones.csv", index=False)
df_validaciones.to_csv(TABLAS_DIR / "validaciones.csv", index=False)
df_metricas.to_csv(TABLAS_DIR / "metricas_internas.csv", index=False)
df_feedback.to_csv(TABLAS_DIR / "feedback.csv", index=False)

print("Tablas exportadas en:", TABLAS_DIR)
for p in sorted(TABLAS_DIR.glob("*.csv")):
    print("-", p.name)

Tablas exportadas en: C:\Users\angel\OneDrive\Escritorio\tfg\resultados_dashboard_notebook\tablas_resumen
- feedback.csv
- iteraciones.csv
- metricas_internas.csv
- validaciones.csv


## 10. Lectura de resultados

- `fase1` puede tener violaciones relacionadas con habitaciones porque su habitación es provisional.
- `fase2` debe mejorar o eliminar `RoomGenderMix`, `PatientRoomCompatibility` y `RoomCapacity`.
- Si aparece `UncoveredRoom`, es esperable mientras la fase 3 de enfermería no esté implementada.
- Si una iteración no genera `solucion_fase2.json`, significa que la fase 2 fue infactible y generó feedback para la siguiente iteración.